In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://knowledge.crewai.com/how-to/Installing-CrewAI/"])
documents = loader.load()

print(f"Total documents {len(documents)}")
documents


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
_chunks = _splitter.split_documents(documents)
print(f"Total chunks {len(_chunks)}")
_chunks

In [ ]:
_chunk_text = [d.page_content for d in _chunks]
_chunk_text

In [ ]:
import numpy as np
from langchain_ollama import OllamaEmbeddings
from crewai.rag.embeddings.factory import build_embedder, get_embedding_function
from com.example.ai.config import _embedder_config_st
from chromadb.utils import embedding_functions

# 1. Initialize the embedding function
# You can specify a different model name or device (e.g., "cuda")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)
embedding_function = build_embedder(_embedder_config_st)

# dimensions=768
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

st_vectors = sentence_transformer_ef.embed_query(input=_chunk_text)
st_array = np.array(st_vectors)
print(f"SentenceTransformer embedding shape {st_array.shape}")
print(st_array[0])

ollama_vectors = embeddings.embed_documents(_chunk_text)
ollama_array = np.array(ollama_vectors)
print(f"Ollama embedding shape {ollama_array.shape}")
print(ollama_array[0])

#### This example walks through building a retrieval augmented generation (RAG) application using Ollama and embedding models.

In [ ]:
# Step 1: Generate embeddings
import os
import chromadb
from chromadb.utils import embedding_functions

## Create a simple txt file
persist_directory = f"{os.getenv("WORK_DIR")}/storage/chromadb"
client = chromadb.PersistentClient(path=persist_directory)

#
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

#
collection = client.get_or_create_collection(name="docs", embedding_function=sentence_transformer_ef)

In [ ]:
# Step 1: Generate embeddings
import ollama
import uuid
from chromadb.utils import embedding_functions

documents = [
  "Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels",
  "Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands",
  "Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall",
  "Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight",
  "Llamas are vegetarians and have very efficient digestive systems",
  "Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old",
]

#
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

st_vectors = sentence_transformer_ef.embed_query(input=documents)
st_array = np.array(st_vectors)
print(f"SentenceTransformer embedding shape {st_array.shape}")

# Prepare data for ChromaDB
ids = []
metadatas = []
documents_text = []
embeddings_list = []

# store each document in a vector embedding database
for i, (doc, embedding) in enumerate(zip(documents, st_vectors)):
  # Generate unique ID
  doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
  ids.append(doc_id)
  
  # Prepare metadata
  metadata = dict()
  metadata['doc_index'] = doc_id
  metadata['content_length'] = len(doc)
  metadatas.append(metadata)
  
  # Document content
  documents_text.append(doc)
  
  # Embedding
  embeddings_list.append(embedding.tolist())

  # mxbai-embed-large
  #response = ollama.embed(model=os.environ["OPENAI_EMBEDDING_MODEL_ID"], input=d)
  #embeddings = response["embeddings"]
  
  # Add to collection
  try:
      collection.add(
          ids=ids,
          embeddings=embeddings_list,
          metadatas=metadatas,
          documents=documents_text
      )
  except Exception as e:
      print(f"Error adding documents to vector store: {e}")
      raise

print(f"Successfully added {len(documents)} documents to vector store")
print(f"Total documents in collection: {collection.count()}")

In [4]:
# Step 2: Retrieve : the most relevant document given an example prompt:

import ollama

query = ["What animals are llamas related to ?"]

st_vectors = sentence_transformer_ef.embed_query(input=query)
st_array = np.array(st_vectors)
print(f"SentenceTransformer embedding shape {st_array.shape}")

# # generate an embedding for the input and retrieve the most relevant doc
# response = ollama.embed(
#   model=os.environ["OPENAI_EMBEDDING_MODEL_ID"],
#   input=prompt
# )
# response["embeddings"]

results = collection.query(
  query_embeddings=st_vectors,
  n_results=1
)

context = results['documents'][0][0]

SentenceTransformer embedding shape (1, 768)


In [5]:
#
# Step 3: Generate : response combining the prompt and data we retrieved in step 2
#
input = "What animals are llamas related to ?"
output = ollama.generate(
  model=os.environ["OPENAI_MODEL_NAME"],
  prompt=f"Using this data: {context}. Respond to this prompt: {input}"
)

print(output['response'])

Llamas are related to vicuñas and camels, as they are all members of the camelid family.


In [8]:
query = ["The sky is blue because of Rayleigh scattering"]

st_vectors = sentence_transformer_ef.embed_query(input=query)
st_array = np.array(st_vectors)
print(st_vectors[0])  # vector length
print(f"SentenceTransformer embedding shape {st_array.shape}")

[ 1.58737060e-02 -8.74826163e-02  4.83948737e-04 -3.94806676e-02
 -1.77200716e-02 -7.97013938e-03 -2.12956760e-02  4.00393736e-03
 -1.63432628e-01  1.57468189e-02 -2.83825807e-02  7.67338723e-02
  4.34484594e-02 -6.96823979e-03  9.25518852e-03 -2.29994915e-02
  2.81557534e-02 -2.04205085e-02 -2.56039985e-02 -2.16660798e-02
 -3.19342781e-03 -6.30690949e-03  2.83747688e-02  5.09061757e-03
  2.69794036e-02 -1.65832974e-02  5.44576300e-03  9.04429052e-03
 -3.15425843e-02  3.86136957e-02 -6.49123266e-02 -4.96983603e-02
 -1.93850533e-03 -3.12396977e-02  1.34024594e-06  2.13339576e-03
 -3.81175205e-02  3.88953015e-02  2.12262031e-02  3.26798968e-02
  1.53258881e-02  3.51413488e-02 -2.12134533e-02  7.33033642e-02
 -3.58207226e-02  7.74400607e-02 -1.47639597e-02  9.22239497e-02
 -7.43212132e-03 -1.25751244e-02 -4.30552941e-03  3.13329138e-02
 -4.41304455e-03  2.30966732e-02  5.94693460e-02  2.35644039e-02
 -3.62671986e-02  3.83057445e-02  9.52288602e-03  9.44222510e-02
 -2.82050222e-02  2.69925

In [6]:
import os
import ollama

single = ollama.embed(
  model=os.environ["OPENAI_EMBEDDING_MODEL_ID"],
  input='The sky is blue because of Rayleigh scattering'
)
print(single['embeddings'][0])  # vector length
print(len(single['embeddings'][0]))  # vector length

[0.023892084, 0.022229966, -0.15269239, -0.010197603, 0.056493305, 0.08489726, -0.0042790417, 0.0046243374, 0.02865354, -0.06537465, 0.035416484, 0.075495146, 0.07535521, 0.07798032, 0.0173951, 0.02618654, 0.0024474177, -0.035946146, 0.0071941707, 0.015974646, -0.06760857, -0.03763745, 0.012154261, -0.056366216, 0.053398967, 0.07398132, -0.015732786, -0.013338346, -0.0020769748, -0.026457911, 0.049680237, -0.037722107, -0.020525176, -0.04244258, -0.02565084, -0.024173232, 0.05399797, -0.002309125, 0.014618152, -0.00234048, 0.022236845, -0.022602083, -0.012697005, -0.013832112, 0.017284725, -0.0065014474, 0.047591258, 0.06283913, -0.032580387, -0.039906736, -0.050770573, 0.06505398, 0.004307632, -0.07331451, 0.03492824, 0.041894276, 0.039532218, -0.030327288, 0.0036885955, 0.016241407, 0.069296464, 0.07423016, 0.030480705, 0.064470015, 0.035091624, -0.050633598, -0.054948118, -0.007076625, 0.0050007687, -0.03635548, 0.07135583, -0.049124554, 0.0022741272, 0.065793045, -0.042549245, -0.0

In [ ]:
from crewai_tools import WebsiteSearchTool
from crewai import Agent, LLM
from crewai.project import agent
from crewai_tools import RagTool
from crewai.tools import tool
from com.example.ai.config import create_rag_config

_tool_config = create_rag_config()

#@tool
def website_search(http_url: str):
    """
    Useful for when you need to ask with lookup on website data.
    """
    #return retriever.get_relevant_documents()
    # Proper Configuration Structure
    search_tool = WebsiteSearchTool(
        website=http_url,
        config=_tool_config
    )

    return search_tool.run()

website_search("https://knowledge.crewai.com/how-to/Installing-CrewAI/")   

In [ ]:
researcher = Agent(
    name="Researcher",
    role="Researches topics by searching the website data.",
    tools=[website_search],
    goal="Answer questions by retrieving relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant specializing in searching and retrieving information from a website. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)

researcher_boss = Agent(
    name="Researcher Boss",
    role="Challenges the researcher to bring out the best out of his findings",
    tools=[website_search],
    goal="Ask further questions to the researcher and validates the retrieved relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant boss and your job is to make sure the retrieved information is correct. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)


In [ ]:
from crewai import Task
research_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher,
    #async_execution=True
)

boss_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher_boss,
    #async_execution=True
)


In [ ]:
from crewai import Crew
# Create Crew
research_crew1 = Crew(
    agents=[researcher, researcher_boss],
    tasks=[research_task, boss_task],
    verbose=True,  # This will print logs to the console as the crew works
    planning=True,
)

# Job Context
job_crew_works = {
    'crewai_url': 'https://knowledge.crewai.com/how-to/Installing-CrewAI/',
    'personal_writeup': "Accomplished Researcher with 18 years of experience, specializing in setting up CrewAI kind of agent based systems"
}

async_result = await research_crew1.kickoff_async(inputs=job_crew_works)
print(async_result)

# Kickoff the Crew's Work
#research_crew1.kickoff(inputs=job_crew_works)
#print(result)